# Electricity Consumption and Bill Prediction

## Machine Learning Notebook

This notebook prepares the household electricity dataset, compares five regression models, tunes model parameters, evaluates tuned models, runs cross-validation, and summarizes the final model.

**Target:** `Monthly_Usage_kWh`  
**Models:** Linear Regression, Polynomial Regression, Decision Tree, Random Forest, and Support Vector Regression (SVR).

## IMPORTING LIBRARIES

In [ ]:
from pathlib import Path
import warnings

import joblib
import matplotlib.pyplot as plt
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, KFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures, StandardScaler
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor

warnings.filterwarnings("ignore")
print("Libraries imported successfully!")

## LOADING DATASET

The project dataset is stored in `data/electricity_bill_prediction.csv`. The path search supports opening this notebook from the project folder, its `ml_model` folder, or the workspace folder.

In [ ]:
possible_project_folders = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd() / "electricity_prediction",
]

PROJECT_DIR = None
for folder in possible_project_folders:
    if (folder / "data" / "electricity_bill_prediction.csv").exists():
        PROJECT_DIR = folder
        break

if PROJECT_DIR is None:
    raise FileNotFoundError(
        "Could not find data/electricity_bill_prediction.csv. "
        "Open the notebook from the project folder or ml_model folder."
    )

DATA_PATH = PROJECT_DIR / "data" / "electricity_bill_prediction.csv"
df = pd.read_csv(DATA_PATH)
print("Dataset path:", DATA_PATH)
df.head()

In [ ]:
print("Number of samples in dataset:", df.shape[0])
print("Number of columns in dataset:", df.shape[1])
print("\nColumn names:")
print(df.columns.tolist())
print("\nMissing values by column:")
print(df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())

### DATA CLEANING

Column names are standardized, duplicate rows are removed, numeric fields are converted to numbers, and rows without a target value are removed. Missing input values are handled later inside the preprocessing pipeline.

In [ ]:
df.columns = df.columns.str.strip()
df = df.drop_duplicates().copy()

numeric_columns = [
    "Household_Size", "Rooms", "AC_Units", "Fan_Count", "Light_Count",
    "Refrigerator", "TV_Count", "Washing_Machine", "Other_Appliances",
    "Previous_Month_Usage_kWh", "Temperature_C", "Monthly_Usage_kWh"
]

for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

df["Area_Type"] = df["Area_Type"].astype("string").str.strip()
df["Area_Type"] = df["Area_Type"].replace("", pd.NA)
df = df.dropna(subset=["Monthly_Usage_kWh"]).copy()

target = "Monthly_Usage_kWh"
feature_columns = [
    "Household_Size", "Rooms", "AC_Units", "Fan_Count", "Light_Count",
    "Refrigerator", "TV_Count", "Washing_Machine", "Other_Appliances",
    "Previous_Month_Usage_kWh", "Temperature_C", "Area_Type"
]

print("Cleaned dataset shape:", df.shape)
print("Duplicate rows remaining:", df.duplicated().sum())
print("Missing target values:", df[target].isnull().sum())
print("Missing input values:")
print(df[feature_columns].isnull().sum())

### BASIC DATA EXPLORATION

In [ ]:
df[feature_columns + [target]].describe(include="all").T

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df[target], bins=20, color="teal", edgecolor="white")
axes[0].set_title("Monthly Electricity Usage")
axes[0].set_xlabel("Usage (kWh)")
axes[0].set_ylabel("Number of households")

area_counts = df["Area_Type"].value_counts(dropna=False)
axes[1].bar(area_counts.index.astype(str), area_counts.values, color="slateblue")
axes[1].set_title("Records by Area Type")
axes[1].set_xlabel("Area type")
axes[1].set_ylabel("Number of households")

plt.tight_layout()
plt.show()

## INDEPENDENT AND DEPENDENT FEATURES

In [ ]:
X = df[feature_columns].copy()
y = df[target].copy()

print("Number of input features:", X.shape[1])
print("Target variable:", target)

## TRAIN TEST SPLIT

The same 80:20 split is used for every model. Imputation, scaling, and category encoding are fitted later using training data only.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=91
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

### PREPROCESSING

Numerical values use median imputation and standard scaling. `Area_Type` uses most-frequent imputation and one-hot encoding. Keeping preprocessing inside each model pipeline ensures that training folds learn their own preprocessing values.

In [ ]:
numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)


def make_preprocessor():
    numeric_steps = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_steps = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoding", OneHotEncoder(handle_unknown="ignore"))
    ])

    return ColumnTransformer([
        ("numeric", numeric_steps, numeric_features),
        ("categorical", categorical_steps, categorical_features)
    ])


def make_model_pipeline(regressor, polynomial=False):
    steps = [("preprocessing", make_preprocessor())]

    if polynomial:
        steps.append(("polynomial", PolynomialFeatures(include_bias=False)))

    steps.append(("regressor", regressor))
    return Pipeline(steps)

## MODEL TRAINING, PREDICTION AND PERFORMANCE

The first comparison uses default model settings. MAE, MSE, and R² are calculated on the same held-out test rows.

In [ ]:
baseline_models = {
    "Linear Regression": make_model_pipeline(LinearRegression()),
    "Polynomial Regression": make_model_pipeline(LinearRegression(), polynomial=True),
    "Decision Tree": make_model_pipeline(DecisionTreeRegressor(random_state=91)),
    "Random Forest": make_model_pipeline(
        RandomForestRegressor(n_estimators=100, random_state=91, n_jobs=-1)
    ),
    "SVR": make_model_pipeline(SVR())
}

baseline_results = {}
baseline_predictions = {}

for name, model in baseline_models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    baseline_predictions[name] = y_pred

    baseline_results[name] = {
        "MAE": mean_absolute_error(y_test, y_pred),
        "MSE": mean_squared_error(y_test, y_pred),
        "R2": r2_score(y_test, y_pred)
    }

    print("\nModel:", name)
    print("Test MAE:", round(baseline_results[name]["MAE"], 3))
    print("Test MSE:", round(baseline_results[name]["MSE"], 3))
    print("Test R2:", round(baseline_results[name]["R2"], 4))

In [ ]:
baseline_results_df = pd.DataFrame(baseline_results).T
baseline_results_df = baseline_results_df.sort_values("MAE")
baseline_results_df.round(4)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, metric, color in zip(
    axes, ["MAE", "MSE", "R2"], ["teal", "darkorange", "slateblue"]
):
    values = baseline_results_df[metric].sort_values(ascending=(metric != "R2"))
    values.plot(kind="bar", ax=ax, color=color)
    ax.set_title("Baseline " + metric + " Comparison")
    ax.set_ylabel(metric)
    ax.tick_params(axis="x", rotation=70)

plt.tight_layout()
plt.show()

## HYPERPARAMETER TUNING

Grid search selects parameters using the training set only. The final test set remains separate for the after-tuning evaluation. Linear Regression has no comparable complexity parameter, so it remains as the untuned reference model.

In [ ]:
parameter_grids = {
    "Polynomial Regression": {
        "polynomial__degree": [1, 2]
    },
    "Decision Tree": {
        "regressor__max_depth": [4, 8, None],
        "regressor__min_samples_leaf": [1, 3]
    },
    "Random Forest": {
        "regressor__n_estimators": [50, 100],
        "regressor__max_depth": [8, None],
        "regressor__min_samples_leaf": [1, 3]
    },
    "SVR": {
        "regressor__kernel": ["rbf"],
        "regressor__C": [1, 10, 100],
        "regressor__epsilon": [0.1, 10]
    }
}

In [ ]:
tuning_folds = KFold(n_splits=5, shuffle=True, random_state=91)
tuned_models = {}
best_parameters = {}

for name, model in baseline_models.items():
    if name in parameter_grids:
        grid = GridSearchCV(
            estimator=model,
            param_grid=parameter_grids[name],
            scoring="neg_mean_absolute_error",
            cv=tuning_folds,
            n_jobs=-1
        )
        grid.fit(X_train, y_train)
        tuned_models[name] = grid.best_estimator_
        best_parameters[name] = grid.best_params_

        print("\nModel:", name)
        print("Best parameters:", grid.best_params_)
        print("Best training CV MAE:", round(-grid.best_score_, 3), "kWh")
    else:
        model.fit(X_train, y_train)
        tuned_models[name] = model
        best_parameters[name] = "No tuning required"
        print("\nModel:", name, "- trained as the reference model")

In [ ]:
parameter_table = pd.DataFrame([
    {"Model": name, "Selected Parameters": str(parameters)}
    for name, parameters in best_parameters.items()
])
parameter_table

### AFTER-TUNING MODEL EVALUATION

In [ ]:
tuned_results = {}
tuned_predictions = {}

for name, model in tuned_models.items():
    y_pred = model.predict(X_test)
    tuned_predictions[name] = y_pred

    tuned_results[name] = {
        "MAE": mean_absolute_error(y_test, y_pred),
        "MSE": mean_squared_error(y_test, y_pred),
        "R2": r2_score(y_test, y_pred)
    }

    print("\nModel:", name)
    print("Test MAE:", round(tuned_results[name]["MAE"], 3))
    print("Test MSE:", round(tuned_results[name]["MSE"], 3))
    print("Test R2:", round(tuned_results[name]["R2"], 4))

In [ ]:
comparison_rows = []

for name in baseline_models:
    for stage, values in [
        ("Before tuning", baseline_results[name]),
        ("After tuning", tuned_results[name])
    ]:
        comparison_rows.append({
            "Model": name,
            "Stage": stage,
            "MAE": values["MAE"],
            "MSE": values["MSE"],
            "R2": values["R2"]
        })

before_after_table = pd.DataFrame(comparison_rows)
before_after_table.round(4)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, metric in zip(axes, ["MAE", "MSE", "R2"]):
    chart_data = before_after_table.pivot(index="Model", columns="Stage", values=metric)
    chart_data.plot(kind="bar", ax=ax, color=["lightgray", "teal"])
    ax.set_title(metric + " Before and After Tuning")
    ax.set_ylabel(metric)
    ax.tick_params(axis="x", rotation=65)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## ACTUAL VS PREDICTED TABLE

The table below shows the first 20 held-out records for all tuned models. The full test prediction table is saved near the end of the notebook.

In [ ]:
actual_vs_predicted = pd.DataFrame({"Actual_kWh": y_test.reset_index(drop=True)})

for name, predictions in tuned_predictions.items():
    actual_vs_predicted[name + "_Predicted_kWh"] = predictions

actual_vs_predicted.head(20).round(2)

## CROSS VALIDATION OF ALL MODELS AFTER TUNING

Five outer folds estimate performance using training data only. For models with hyperparameters, grid search is repeated inside each outer training fold. This lets each outer validation fold remain separate from parameter selection. Linear Regression is cross-validated directly because it has no tuning grid.

In [ ]:
outer_folds = KFold(n_splits=5, shuffle=True, random_state=42)
inner_folds = KFold(n_splits=3, shuffle=True, random_state=91)
cv_scoring = {
    "MAE": "neg_mean_absolute_error",
    "MSE": "neg_mean_squared_error",
    "R2": "r2"
}

cv_results = {}

for name, model in baseline_models.items():
    if name in parameter_grids:
        cv_model = GridSearchCV(
            estimator=model,
            param_grid=parameter_grids[name],
            scoring="neg_mean_absolute_error",
            cv=inner_folds,
            n_jobs=1
        )
    else:
        cv_model = model

    scores = cross_validate(
        cv_model,
        X_train,
        y_train,
        cv=outer_folds,
        scoring=cv_scoring,
        n_jobs=1
    )

    cv_results[name] = {
        "CV MAE Mean": -scores["test_MAE"].mean(),
        "CV MAE SD": scores["test_MAE"].std(ddof=1),
        "CV MSE Mean": -scores["test_MSE"].mean(),
        "CV MSE SD": scores["test_MSE"].std(ddof=1),
        "CV R2 Mean": scores["test_R2"].mean(),
        "CV R2 SD": scores["test_R2"].std(ddof=1)
    }

    print("Completed:", name)

In [ ]:
cv_summary = pd.DataFrame(cv_results).T
cv_summary = cv_summary.sort_values("CV MAE Mean", kind="stable")
cv_summary.round(4)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, metric, lower_is_better, color in zip(
    axes,
    ["MAE", "MSE", "R2"],
    [True, True, False],
    ["teal", "darkorange", "slateblue"]
):
    mean_values = cv_summary["CV " + metric + " Mean"]
    std_values = cv_summary["CV " + metric + " SD"]
    mean_values.plot(kind="bar", yerr=std_values, capsize=4, ax=ax, color=color)
    ax.set_title("5-Fold CV " + metric)
    ax.set_ylabel(metric)
    ax.tick_params(axis="x", rotation=70)

plt.tight_layout()
plt.show()

## FINAL MODEL AND CONCLUSION

The final model is selected by the lowest mean cross-validation MAE. If models tie, the table's original order favors the simpler Linear Regression. The held-out test metrics are shown for this selected model.

In [ ]:
best_model_name = cv_summary.index[0]
best_model = tuned_models[best_model_name]
best_test_metrics = tuned_results[best_model_name]

print("Best model based on cross-validation MAE:", best_model_name)
print("Selected parameters:", best_parameters[best_model_name])
print("Held-out test MAE:", round(best_test_metrics["MAE"], 3), "kWh")
print("Held-out test MSE:", round(best_test_metrics["MSE"], 3), "kWh squared")
print("Held-out test R2:", round(best_test_metrics["R2"], 4))

In [ ]:
best_predictions = tuned_predictions[best_model_name]
residuals = y_test.to_numpy() - best_predictions
minimum_value = min(y_test.min(), best_predictions.min())
maximum_value = max(y_test.max(), best_predictions.max())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(y_test, best_predictions, alpha=0.7, color="teal")
axes[0].plot(
    [minimum_value, maximum_value],
    [minimum_value, maximum_value],
    color="red", linestyle="--", label="Perfect prediction"
)
axes[0].set_title(best_model_name + " - Actual vs Predicted")
axes[0].set_xlabel("Actual usage (kWh)")
axes[0].set_ylabel("Predicted usage (kWh)")
axes[0].legend()

axes[1].scatter(best_predictions, residuals, alpha=0.7, color="slateblue")
axes[1].axhline(0, color="red", linestyle="--")
axes[1].set_title(best_model_name + " - Residual Plot")
axes[1].set_xlabel("Predicted usage (kWh)")
axes[1].set_ylabel("Actual - predicted (kWh)")

plt.tight_layout()
plt.show()

### SAVING MODEL AND RESULTS

The selected model includes its preprocessing steps. It is saved as a separate notebook model artifact for later backend integration.

In [ ]:
RESULTS_DIR = PROJECT_DIR / "results" / "notebook"
MODELS_DIR = PROJECT_DIR / "models"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

baseline_results_df.to_csv(RESULTS_DIR / "baseline_model_comparison.csv")
before_after_table.to_csv(RESULTS_DIR / "before_after_tuning.csv", index=False)
cv_summary.to_csv(RESULTS_DIR / "cross_validation_comparison.csv")
actual_vs_predicted.to_csv(RESULTS_DIR / "actual_vs_predicted.csv", index=False)

model_path = MODELS_DIR / "electricity_usage_best_from_notebook.joblib"
joblib.dump(best_model, model_path)

print("Saved model:", model_path)
print("Saved tables:", RESULTS_DIR)